# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, their fields, and column IDs.

All entities are referenced by their Croissant `@id` value. This ensures correct referencing throughout the notebook.

Let's inspect the available record sets and their corresponding fields:

In [ ]:
# List available record sets
record_sets = [rs for rs in metadata.record_sets]
print('Record sets found in the dataset:')
for rs in record_sets:
    print(f" - {rs.id}: {rs.name}")

# For each record set, print its fields/columns
for rs in record_sets:
    print(f"\nRecord Set: {rs.id} ({rs.name})")
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for f in rs.fields:
            print(f"    - {f.id}: {f.name} (data_type={f.data_type if hasattr(f, 'data_type') else 'N/A'})")
    elif hasattr(rs, 'columns') and rs.columns:
        print('  Columns:')
        for c in rs.columns:
            print(f"    - {c.id}: {c.name} (data_type={c.data_type if hasattr(c, 'data_type') else 'N/A'})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> **Note:** We use the `@id` of the record set and columns as shown above for extraction.

In [ ]:
# List all the record set ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for Record Set: {record_set_id}")
    # Convert records generator to DataFrame
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"  Loaded {len(records)} records. Columns: {dataframes[record_set_id].columns.tolist()}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  ERROR loading records: {e}")

# Choose a record set for demonstration (pick the first with data)
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"\nPreview of record set {main_record_set_id}:\n", dataframes[main_record_set_id].head())
else:
    print("No record set with records found.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps:
- Filtering records based on a numeric field.
- Normalizing a numeric field.
- Grouping by a categorical (string) field.

All field references use their Croissant `@id` (column ID) for consistency.

In [ ]:
import numpy as np

# Choose a numeric field and a group field for exploration
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Attempt to automatically detect a likely numeric field
    numeric_candidate = None
    group_candidate = None
    for col in df.columns:
        if df[col].dtype in [np.float64, np.int64, float, int] or pd.api.types.is_numeric_dtype(df[col]):
            if numeric_candidate is None:
                numeric_candidate = col
        else:
            if group_candidate is None and pd.api.types.is_object_dtype(df[col]):
                group_candidate = col

    print(f"Numeric field detected: {numeric_candidate}")
    print(f"Group field detected: {group_candidate}")

    # Filter records (if numeric field present)
    if numeric_candidate:
        # Ensure values are numeric
        df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')
        threshold = df[numeric_candidate].quantile(0.5)  # use median as threshold
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with {numeric_candidate} > {threshold} (median):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_candidate}_normalized"] = (
            (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) /
            filtered_df[numeric_candidate].std()
        )
        print(f"Normalized {numeric_candidate} for filtered records:")
        print(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

        # Group by a field (if group candidate present)
        if group_candidate and group_candidate in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_candidate).mean(numeric_only=True)
            print(f"Grouped data by {group_candidate}:")
            print(grouped_df.head())
else:
    print("No main record set with usable numeric or group fields found.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below, we create a histogram of the numeric field and, if available, a box plot grouped by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_candidate:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_candidate].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f'Distribution of {numeric_candidate}')
    plt.xlabel(numeric_candidate)
    plt.ylabel('Frequency')
    plt.show()

    if group_candidate and group_candidate in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_candidate, y=numeric_candidate, data=df)
        plt.title(f'{numeric_candidate} by {group_candidate}')
        plt.xlabel(group_candidate)
        plt.ylabel(numeric_candidate)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we explored the metadata and main record sets of the dataset using the `mlcroissant` library, then loaded records and performed filtering, normalization, and grouping using column `@id` references. We visualized distributions and group differences to enable further analysis of mass spectrometry-based protein measurements.

For deeper analysis, further domain-specific transformations can be applied based on your research questions.